# Chapter 20: Cayley-Klein Geometries

**Source span.** Sections 20.1-20.7, printed pp. 375-398 / PDF pp. 397-420.

**Chapter goal.** Build the Cayley-Klein idea computationally: choose an absolute conic, use it to measure with cross-ratios, and read Euclidean, hyperbolic, elliptic, pseudo-Euclidean, and degenerate geometries as different projective choices of that conic.

The chapter starts by revisiting the circular points `I` and `J`. In the Euclidean story they looked special; in the Cayley-Klein story they are one degenerate absolute among many. The notebook therefore treats metric language as a derived projective construction. A line meets the absolute in two reference points, a pencil from a point has two tangent reference lines, and a logarithm turns the cross-ratio into an oriented measurement.

The source is used here only for orientation and coverage. The prose, code, diagrams, tables, and checks are original and standalone.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives-on-Projective-Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-20-cayley-klein-geometries"
for child in ("figures", "html", "tables", "checks"):
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

ARTIFACT_ROOT.relative_to(BOOK_ROOT).as_posix()


## Computational Translation Guide

| Book idea | Computational object in this notebook | What is checked |
| --- | --- | --- |
| Absolute or fundamental conic `F` | A symmetric homogeneous quadratic form, usually `x^2 + y^2 - z^2 = 0` or a degenerate pair | Line intersections, tangent/polar equations, and signature data |
| Points `X, Y` where a line meets `F` | Roots of a quadratic on the projective line | Real distinct, complex conjugate, or repeated root classification |
| Distance between `p` and `q` | `c_dist * log((p,q;X,Y))` | Additivity and antisymmetry from cross-ratio identities |
| Angle between two lines | The dual construction using the two tangents from their meet to `F` | Agreement with Laguerre's Euclidean formula when `F` is `I,J` |
| Degenerate line measurement | A limit of nondegenerate absolutes, compared to a chosen unit | Relative limit `f_a(alpha,q) -> q/a` |
| Classification | Real projective equivalence class of the conic pair | Seven type census plus distance/angle type grid |

The important habit is to avoid inserting Euclidean distance by hand. In each cell below the metric quantity is recovered from projective data and then compared with an invariant that would fail if the absolute conic were changed incorrectly.


In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import sympy as sp
from plotly.subplots import make_subplots

from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json, save_table
from utils.conics import polar_line
from utils.projective import hpoint

plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 160, "font.size": 10})

COLORS = {
    "blue": "#276fbf",
    "green": "#2a9d55",
    "orange": "#d9822b",
    "red": "#c43b3b",
    "purple": "#7b61a8",
    "gray": "#4b5563",
    "teal": "#168a8f",
}

def rel(path):
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()

def fig_path(name):
    return ARTIFACT_ROOT / "figures" / name

def html_path(name):
    return ARTIFACT_ROOT / "html" / name

def complex_cross_ratio(p, q, X, Y):
    p, q, X, Y = map(complex, (p, q, X, Y))
    return ((p - X) * (q - Y)) / ((p - Y) * (q - X))

def hyperbolic_line_distance(p, q):
    return -0.5 * np.log(complex_cross_ratio(p, q, -1, 1))

def elliptic_line_distance(p, q):
    return (1 / (2j)) * np.log(complex_cross_ratio(p, q, -1j, 1j))

def euclidean_angle_from_absolute(slope_l, slope_m):
    return (1 / (2j)) * np.log(complex_cross_ratio(slope_l, slope_m, -1j, 1j))

def parabolic_relative_ratio(alpha, q, unit):
    s = np.sqrt(alpha + 0j)
    num = np.log(q * s - 1) - np.log(-q * s - 1)
    den = np.log(unit * s - 1) - np.log(-unit * s - 1)
    return num / den

def line_unit_circle_roots(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    d = q - p
    A = float(d @ d)
    B = float(2 * p @ d)
    C = float(p @ p - 1)
    roots = np.roots([A, B, C])
    return roots, B * B - 4 * A * C

def classify_line_measurement(p, q, tol=1e-9):
    roots, disc = line_unit_circle_roots(p, q)
    if abs(disc) < tol:
        return "parabolic tangent", disc, None
    if disc < 0:
        return "elliptic line", disc, None
    roots = sorted(float(np.real(t)) for t in roots)
    cr = complex_cross_ratio(0, 1, roots[0], roots[1])
    if abs(cr.imag) < 1e-8 and cr.real > 0:
        return "real hyperbolic", disc, cr.real
    return "separated by absolute", disc, cr.real if abs(cr.imag) < 1e-8 else cr

def klein_distance_inside(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    num = 1 - float(p @ q)
    den = math.sqrt((1 - float(p @ p)) * (1 - float(q @ q)))
    return math.acosh(max(1.0, num / den))

F_UNIT_CIRCLE = np.diag([1.0, 1.0, -1.0])
generated_paths = []
checks = {}


## Chapter Storyboard

The visual sequence follows the chapter's logic rather than the old generic scaffold.

1. **`I` and `J` as a degenerate absolute.** The Euclidean circular points become a dual conic; Laguerre's angle formula is a Cayley-Klein angle measurement.
2. **One projective line, three measurement behaviors.** Real distinct, complex conjugate, and coincident absolute points produce hyperbolic, elliptic, and parabolic line measurements.
3. **Invariant proof scaffold.** Cross-ratio identities explain zero distance, antisymmetry, and additivity before any model is named.
4. **A real planar absolute.** The unit conic gives a single plane where real, imaginary, mixed, and tangent measurements can all be seen.
5. **Census of Cayley-Klein planes.** The seven real projective conic types explain elliptic, hyperbolic, Euclidean, pseudo-Euclidean, and fully degenerate cases.
6. **Model comparison lab.** Euclidean, hyperbolic, elliptic, and degenerate models are displayed side by side with their visible absolute data.


In [ ]:
storyboard = {
    "chapter goal": "Use an absolute conic to derive distance and angle measurements, then classify the resulting Cayley-Klein geometries.",
    "source span read": "Perspectives on Projective Geometry, Chapter 20, sections 20.1-20.7; printed pp. 375-398 / PDF pp. 397-420.",
    "concept inventory": [
        "I and J as a degenerate dual conic and the line at infinity as the primal absolute",
        "distance from intersections of a line with the absolute conic",
        "angle from tangents from a point to the absolute conic",
        "hyperbolic, elliptic, and parabolic line measurements from cross-ratio roots",
        "parabolic measurement as a relative limit with a chosen unit length",
        "real planar conic model with interior, exterior, tangent, and separated cases",
        "seven real projective conic types and coarser/finer classifications",
    ],
    "library routing table": [
        {"concept": "homogeneous coordinates and conics", "representation": "matrices, roots, cross-ratios", "library": "numpy + local utils", "why": "small transparent projective operations", "fallback": "manual linear algebra"},
        {"concept": "symbolic cross-ratio laws", "representation": "exact rational identities", "library": "sympy", "why": "proof scaffold without numerical branch issues", "fallback": "numeric sampling"},
        {"concept": "proof route", "representation": "directed invariant graph", "library": "networkx + matplotlib", "why": "shows which construction feeds each property", "fallback": "markdown table"},
        {"concept": "model comparison", "representation": "2D/3D interactive panels", "library": "plotly", "why": "elliptic sphere and model comparisons benefit from rotation and hover", "fallback": "static panels"},
    ],
    "visual sequence": [
        {"artifact_filename": "ij-revisited-absolute-conic.png", "concept": "I/J revisited", "inspection_target": "Compare real slopes with the complex absolute slopes plus/minus i.", "validation": "Laguerre cross-ratio equals arctan slope difference."},
        {"artifact_filename": "invariant-reparametrization-check.png", "concept": "line measurement trichotomy", "inspection_target": "Watch hyperbolic blow-up, elliptic wrap, and parabolic normalization.", "validation": "additivity, unit-modulus elliptic cross-ratio, parabolic limit residual."},
        {"artifact_filename": "visual-route-map.png", "concept": "proof dependencies", "inspection_target": "Trace absolute conic to roots/tangents to cross-ratio to additive measurement.", "validation": "SymPy cross-ratio identities."},
        {"artifact_filename": "absolute-conic-measurement-lab.png", "concept": "planar real conic", "inspection_target": "Distinguish secant, tangent, exterior, and separated line cases.", "validation": "quadratic discriminants and polar incidence."},
        {"artifact_filename": "cayley-klein-classification-census.png", "concept": "seven-type census", "inspection_target": "Read which conic type forces which distance/angle behavior.", "validation": "classification table contains all seven types."},
        {"artifact_filename": "euclidean-hyperbolic-elliptic-degenerate-comparison.html", "concept": "model comparison", "inspection_target": "Compare visible absolutes and equal-step behavior across four models.", "validation": "HTML artifact exists and model table lists the same cases."},
    ],
    "computational checks": [
        "Laguerre formula residual for Euclidean angle from I and J",
        "cross-ratio antisymmetry and multiplicativity symbolic identities",
        "hyperbolic line distance additivity",
        "elliptic cross-ratio unit modulus",
        "parabolic relative limit q/a",
        "unit-circle line discriminant classifications",
        "artifact existence, size, and nonblank raster statistics",
    ],
    "proof-visualization strategy": "Use a NetworkX dependency graph plus SymPy cross-ratio identities; use limiting plots for the parabolic proof move.",
    "risks": "Logarithm branches are kept to small real intervals for additive numeric checks; global multivalued behavior is discussed rather than hidden.",
}
visual_targets = [
    {"artifact": item["artifact_filename"], "concept": item["concept"], "inspection_target": item["inspection_target"], "validation": item["validation"]}
    for item in storyboard["visual sequence"]
]
storyboard_path = save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")
visual_targets_path = save_table(visual_targets, ARTIFACT_ROOT, "tables", "visual-observation-targets.csv")
generated_paths.extend([storyboard_path, visual_targets_path])
[rel(storyboard_path), rel(visual_targets_path)]


## `I` and `J` Revisited: Euclidean Angles Are Already Cayley-Klein

For ordinary Euclidean angle measurement, the absolute is degenerate. In primal form it is the double line at infinity, `z^2 = 0`. In dual form it is the conjugate pair of circular points encoded by `a^2 + b^2 = 0`, so its two directions on the slope line are `-i` and `i`.

That means Laguerre's formula is not an isolated trick. It is the angle half of the Cayley-Klein recipe: two real lines through a point are measured by a cross-ratio against the two absolute tangents through that point. In the affine slope coordinate this becomes a cross-ratio of real slopes with the two complex reference slopes.


In [ ]:
def make_ij_figure():
    m_l, m_m = 0.2, 1.4
    angle_ck = euclidean_angle_from_absolute(m_l, m_m)
    angle_euclid = math.atan(m_m) - math.atan(m_l)
    residual = abs(angle_ck.real - angle_euclid) + abs(angle_ck.imag)

    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5), constrained_layout=True)
    ax = axes[0]
    x = np.linspace(-2.3, 2.3, 200)
    p = np.array([-0.15, -0.05])
    for slope, color, label in [(m_l, COLORS["blue"], "line l"), (m_m, COLORS["orange"], "line m")]:
        ax.plot(x, p[1] + slope * (x - p[0]), color=color, lw=2.2, label=label)
    ax.scatter([p[0]], [p[1]], color="black", zorder=5)
    ax.text(p[0] + 0.05, p[1] - 0.18, "meet p", fontsize=9)
    ax.axhline(1.9, color=COLORS["gray"], lw=1.5, ls="--")
    ax.text(-2.25, 1.98, "primal absolute: z=0, double line at infinity", color=COLORS["gray"], fontsize=9)
    ax.set_xlim(-2.35, 2.35); ax.set_ylim(-1.45, 2.25)
    ax.set_aspect("equal", adjustable="box"); ax.grid(True, alpha=0.18)
    ax.set_title("Real affine lines; absolute is not a visible circle")
    ax.legend(loc="lower right", frameon=False)

    ax = axes[1]
    ax.axhline(0, color="#222222", lw=1.0); ax.axvline(0, color="#222222", lw=1.0)
    ax.scatter([m_l, m_m], [0, 0], s=80, color=[COLORS["blue"], COLORS["orange"]], zorder=4)
    ax.scatter([0, 0], [-1, 1], s=90, color=[COLORS["purple"], COLORS["green"]], zorder=4)
    ax.text(m_l + 0.05, 0.08, "slope l", color=COLORS["blue"])
    ax.text(m_m + 0.05, 0.08, "slope m", color=COLORS["orange"])
    ax.text(0.08, 1.03, "J = i", color=COLORS["green"])
    ax.text(0.08, -1.12, "I = -i", color=COLORS["purple"])
    ax.plot([m_l, 0, m_m, 0, m_l], [0, 1, 0, -1, 0], color="#555555", alpha=0.45, lw=1.2)
    ax.set_xlim(-0.5, 1.75); ax.set_ylim(-1.35, 1.35)
    ax.set_aspect("equal", adjustable="box"); ax.grid(True, alpha=0.18)
    ax.set_title("Dual absolute on the complex slope line")
    ax.set_xlabel("real slope"); ax.set_ylabel("imaginary slope")
    ax.text(-0.45, -1.28, f"(1/2i) log(l,m;I,J) = {angle_ck.real:.6f}\narctan difference = {angle_euclid:.6f}", fontsize=9)

    fig.suptitle("Euclidean angle as a Cayley-Klein measurement from the degenerate absolute", fontsize=13)
    path = fig_path("ij-revisited-absolute-conic.png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path, {"laguerre_angle_residual": float(residual), "angle_radians": float(angle_ck.real)}

ij_path, ij_checks = make_ij_figure()
checks.update(ij_checks)
generated_paths.append(ij_path)
rel(ij_path), ij_checks


## Measurements Along One Line

On a fixed projective line, the absolute contributes two reference points `X` and `Y`. The same formula then has three qualitatively different readings.

- If `X` and `Y` are real and distinct, the open interval between them behaves hyperbolically; equal intrinsic steps get smaller in an affine drawing and the boundary is infinitely far away.
- If `X` and `Y` are complex conjugates, the cross-ratio lies on the unit circle; the logarithm is imaginary and a factor `1/(2i)` turns it into a real elliptic measurement.
- If `X = Y`, the raw cross-ratio gives zero. The useful measurement is relative: compare to a chosen unit before taking the limit as the two absolute points coalesce.


In [ ]:
def make_line_measurement_figure():
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)

    xs = np.linspace(-0.985, 0.985, 500)
    axes[0].plot(xs, np.arctanh(xs), color=COLORS["green"], lw=2.2)
    pts = np.tanh(np.arange(-4, 5) * 0.55)
    axes[0].scatter(pts, np.zeros_like(pts), color=COLORS["orange"], s=28, zorder=4)
    for val in [-1, 1]:
        axes[0].axvline(val, color=COLORS["gray"], ls="--", lw=1)
    axes[0].set_title("Real distinct X,Y: hyperbolic")
    axes[0].set_xlabel("q in (-1,1), p=0"); axes[0].set_ylabel("dist_hyp(0,q)")
    axes[0].grid(True, alpha=0.2)

    xe = np.linspace(-5, 5, 600)
    axes[1].plot(xe, np.arctan(xe), color=COLORS["purple"], lw=2.2)
    ell_pts = np.tan(np.arange(-3, 4) * 0.45)
    axes[1].scatter(ell_pts, np.zeros_like(ell_pts), color=COLORS["teal"], s=28, zorder=4)
    axes[1].axhline(math.pi / 2, color=COLORS["gray"], ls="--", lw=1)
    axes[1].axhline(-math.pi / 2, color=COLORS["gray"], ls="--", lw=1)
    axes[1].set_title("Complex conjugate X,Y: elliptic")
    axes[1].set_xlabel("real projective coordinate q"); axes[1].set_ylabel("dist_ell(0,q)")
    axes[1].grid(True, alpha=0.2)

    a = 0.6
    qvals = np.linspace(-1.15, 1.15, 300)
    axes[2].plot(qvals, qvals / a, color="black", lw=2, label="limit q/a")
    for alpha, color in [(1e-2, COLORS["blue"]), (4e-4, COLORS["orange"]), (1e-6, COLORS["green"])]:
        vals = np.array([parabolic_relative_ratio(alpha, q, a).real for q in qvals])
        axes[2].plot(qvals, vals, color=color, lw=1.8, label=f"alpha={alpha:g}")
    axes[2].set_title("Coincident X=Y: parabolic limit")
    axes[2].set_xlabel("q, with unit a=0.6"); axes[2].set_ylabel("relative measurement")
    axes[2].grid(True, alpha=0.2); axes[2].legend(frameon=False, fontsize=8)

    fig.suptitle("One cross-ratio formula; three line-measurement behaviors", fontsize=13)
    path = fig_path("invariant-reparametrization-check.png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

line_path = make_line_measurement_figure()
generated_paths.append(line_path)

p, q, r = -0.4, 0.1, 0.55
additivity_residual = abs((hyperbolic_line_distance(p, q) + hyperbolic_line_distance(q, r) - hyperbolic_line_distance(p, r)).real)
anti_residual = abs((hyperbolic_line_distance(p, q) + hyperbolic_line_distance(q, p)).real)
ell_moduli = [abs(abs(complex_cross_ratio(0, x, -1j, 1j)) - 1) for x in [-3, -0.7, 0.2, 1.6, 4.0]]
limit_residual = abs(parabolic_relative_ratio(1e-7, 0.7, 0.4).real - 0.7 / 0.4)

line_sample_rows = []
for qv in [-0.75, -0.25, 0.25, 0.75]:
    line_sample_rows.append({
        "q": qv,
        "hyperbolic_dist_0q": float(hyperbolic_line_distance(0, qv).real),
        "elliptic_dist_0q": float(elliptic_line_distance(0, qv).real),
        "parabolic_ratio_alpha_1e-6_unit_0.6": float(parabolic_relative_ratio(1e-6, qv, 0.6).real),
    })
line_samples_path = save_table(line_sample_rows, ARTIFACT_ROOT, "tables", "line-measurement-samples.csv")
generated_paths.append(line_samples_path)
checks.update({
    "line_additivity_residual": float(additivity_residual),
    "line_antisymmetry_residual": float(anti_residual),
    "elliptic_cross_ratio_unit_modulus_error": float(max(ell_moduli)),
    "parabolic_limit_residual": float(limit_residual),
})
rel(line_path), rel(line_samples_path), {k: checks[k] for k in ["line_additivity_residual", "parabolic_limit_residual"]}


## Proof Scaffold: Why the Logarithm Gives an Oriented Scale

The line measurement theorem is a cross-ratio theorem with a logarithm applied afterward. The exact identities below are the computational form of the proof:

`(p,p;X,Y)=1`, `(p,q;X,Y)(q,p;X,Y)=1`, and `(p,q;X,Y)(q,r;X,Y)=(p,r;X,Y)`.

The graph artifact records the dependency chain. The absolute conic is not merely a background drawing: it supplies the two reference elements that make the cross-ratio meaningful.


In [ ]:
def make_dependency_graph():
    graph = nx.DiGraph()
    nodes = {
        "F": "absolute conic F",
        "line": "join p,q or pencil at p",
        "roots": "intersections X,Y\nor tangents X,Y",
        "cr": "cross-ratio\n(p,q;X,Y)",
        "log": "c * log(cross-ratio)",
        "oriented": "zero, sign, additivity\nmod log period",
        "signature": "real projective\nsignature of F",
        "census": "CK geometry type",
        "limit": "coalescing X,Y\nwith a unit",
        "parabolic": "parabolic / Euclidean\nrelative scale",
    }
    for key, label in nodes.items():
        graph.add_node(key, label=label)
    graph.add_edges_from([
        ("F", "roots"), ("line", "roots"), ("roots", "cr"), ("cr", "log"), ("log", "oriented"),
        ("F", "signature"), ("signature", "census"), ("roots", "limit"), ("limit", "parabolic"),
        ("parabolic", "census"), ("oriented", "census"),
    ])
    pos = {"F": (0.0, 0.8), "line": (0.0, -0.35), "roots": (1.6, 0.25), "cr": (3.0, 0.25), "log": (4.35, 0.25), "oriented": (5.95, 0.25), "signature": (1.6, 1.25), "census": (5.95, 1.25), "limit": (3.0, -0.85), "parabolic": (4.65, -0.85)}
    fig, ax = plt.subplots(figsize=(11.5, 4.6), constrained_layout=True)
    node_colors = ["#dbeafe", "#e0f2fe", "#dcfce7", "#fef3c7", "#fde68a", "#fce7f3", "#ede9fe", "#f3f4f6", "#fee2e2", "#ffedd5"]
    nx.draw_networkx_edges(graph, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=16, width=1.4, edge_color="#555555")
    nx.draw_networkx_nodes(graph, pos, ax=ax, node_color=node_colors, node_size=2300, edgecolors="#333333", linewidths=0.8)
    nx.draw_networkx_labels(graph, pos, labels=nx.get_node_attributes(graph, "label"), ax=ax, font_size=8)
    ax.set_title("Proof dependency map for Cayley-Klein measurement")
    ax.axis("off")
    path = fig_path("visual-route-map.png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path

p_sym, q_sym, r_sym, X_sym, Y_sym = sp.symbols("p q r X Y")
def cr_sym(a, b):
    return ((a - X_sym) * (b - Y_sym)) / ((a - Y_sym) * (b - X_sym))

zero_identity = sp.simplify(cr_sym(p_sym, p_sym) - 1)
symmetry_identity = sp.simplify(cr_sym(p_sym, q_sym) * cr_sym(q_sym, p_sym) - 1)
additive_identity = sp.simplify(cr_sym(p_sym, q_sym) * cr_sym(q_sym, r_sym) - cr_sym(p_sym, r_sym))

route_path = make_dependency_graph()
generated_paths.append(route_path)
checks.update({
    "sympy_zero_distance_identity": str(zero_identity),
    "sympy_symmetry_identity": str(symmetry_identity),
    "sympy_additivity_identity": str(additive_identity),
})
rel(route_path), checks["sympy_additivity_identity"]


## A Real Planar Absolute: One Conic, Several Kinds of Measurement

Take the fundamental conic `F: x^2 + y^2 - z^2 = 0`, the unit circle in the affine chart. A single planar model now contains several line behaviors.

A secant line gives real absolute points. If `p` and `q` are not separated by those absolute points, the measurement is real hyperbolic. A line missing the circle gives complex conjugate absolute points and an elliptic line measurement. A tangent line is the parabolic boundary case. If `p` and `q` lie on opposite sides of the absolute, the cross-ratio is negative and the logarithm has an unavoidable imaginary part.

The right panel uses the Klein disk distance formula to draw constant-distance curves from one interior point. They are conics in projective coordinates, not arbitrary analytic decorations.


In [ ]:
def draw_extended_line(ax, p, q, color, label, lw=2.0, ls="-"):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    d = q - p
    ts = np.linspace(-1.0, 2.0, 80)
    pts = p[None, :] + ts[:, None] * d[None, :]
    ax.plot(pts[:, 0], pts[:, 1], color=color, lw=lw, ls=ls, label=label)

def make_planar_conic_figure():
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2), constrained_layout=True)
    theta = np.linspace(0, 2 * math.pi, 500)
    circle_x, circle_y = np.cos(theta), np.sin(theta)

    ax = axes[0]
    ax.plot(circle_x, circle_y, color="black", lw=2.0, label="absolute F")
    scenarios = [
        ("inside-inside", (-0.55, -0.25), (0.48, 0.28), COLORS["green"]),
        ("inside-outside", (-0.15, 0.68), (1.45, 0.92), COLORS["red"]),
        ("outside secant", (-1.45, -0.65), (1.4, -0.35), COLORS["orange"]),
        ("outside exterior", (1.35, -1.1), (1.55, 1.1), COLORS["purple"]),
        ("tangent", (1.0, -1.15), (1.0, 1.15), COLORS["blue"]),
    ]
    scenario_rows = []
    for name, p, q, color in scenarios:
        kind, disc, cr_value = classify_line_measurement(p, q)
        draw_extended_line(ax, p, q, color, f"{name}: {kind}", ls="--" if "tangent" in kind else "-")
        ax.scatter([p[0], q[0]], [p[1], q[1]], color=color, s=28, zorder=5)
        ax.text(q[0] + 0.04, q[1] + 0.04, name, color=color, fontsize=8)
        scenario_rows.append({"case": name, "classification": kind, "discriminant": float(disc), "cross_ratio_real_part": None if cr_value is None else float(np.real(cr_value))})
    ax.set_aspect("equal", adjustable="box"); ax.set_xlim(-1.8, 1.8); ax.set_ylim(-1.45, 1.45)
    ax.grid(True, alpha=0.18); ax.set_title("Distance type from how a line meets the absolute")
    ax.legend(loc="lower left", fontsize=7, frameon=True)

    ax = axes[1]
    ax.plot(circle_x, circle_y, color="black", lw=2.0)
    base = np.array([0.28, -0.18])
    xs = np.linspace(-0.98, 0.98, 280)
    ys = np.linspace(-0.98, 0.98, 280)
    X, Y = np.meshgrid(xs, ys)
    norm2 = X * X + Y * Y
    Qdot = base[0] * X + base[1] * Y
    den = np.sqrt(np.maximum(1e-12, (1 - base @ base) * (1 - norm2)))
    D = np.arccosh(np.maximum(1, (1 - Qdot) / den))
    D = np.ma.masked_where(norm2 >= 0.97**2, D)
    contour = ax.contour(X, Y, D, levels=[0.45, 0.85, 1.25, 1.65, 2.05], colors=[COLORS["blue"], COLORS["green"], COLORS["orange"], COLORS["red"], COLORS["purple"]], linewidths=1.7)
    ax.clabel(contour, inline=True, fontsize=8, fmt="d=%.2g")
    ax.scatter([base[0]], [base[1]], color="black", s=35, zorder=5)
    ax.text(base[0] + 0.04, base[1] - 0.08, "base p", fontsize=9)
    ax.set_aspect("equal", adjustable="box"); ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
    ax.grid(True, alpha=0.18); ax.set_title("Constant real distance curves inside the Klein disk")

    fig.suptitle("Planar Cayley-Klein geometry from the real conic x^2+y^2-z^2=0", fontsize=13)
    path = fig_path("absolute-conic-measurement-lab.png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path, scenario_rows

planar_path, scenario_rows = make_planar_conic_figure()
scenario_table_path = save_table(scenario_rows, ARTIFACT_ROOT, "tables", "planar-line-scenarios.csv")
generated_paths.extend([planar_path, scenario_table_path])

klein_origin_residual = abs(klein_distance_inside(np.array([0.0, 0.0]), np.array([0.35, 0.0])) - math.atanh(0.35))
p_out = hpoint(1.6, 0.25)
polar = polar_line(F_UNIT_CIRCLE, p_out)
a, b, c = polar
if abs(b) > 1e-12:
    xs = np.roots([1 + (a / b) ** 2, 2 * a * c / (b * b), (c / b) ** 2 - 1])
    polar_pts = [hpoint(float(x), float((-a * x - c) / b)) for x in xs]
else:
    x = -c / a
    ys = np.roots([1, 0, x * x - 1])
    polar_pts = [hpoint(float(x), float(y)) for y in ys]
polar_incidence_error = max(abs(float(pt @ F_UNIT_CIRCLE @ pt)) + abs(float(pt @ polar)) for pt in polar_pts)
checks.update({
    "klein_origin_formula_residual": float(klein_origin_residual),
    "polar_absolute_incidence_error": float(polar_incidence_error),
    "planar_line_cases": scenario_rows,
})
rel(planar_path), rel(scenario_table_path), scenario_rows


## Census: The Conic Type Decides the Geometry

Because Cayley-Klein measurements use only projective constructions, real projective equivalence of the absolute conic is the main classifier. The seven real conic-pair types split into nondegenerate, singly degenerate, and doubly degenerate cases. Changing the constants rescales or changes which values are regarded as real, but it does not change the projective source of the measurement.

The census below keeps two classifications visible at once. The seven rows record the conic type. The 3-by-3 grid records the coarser measurement behavior: hyperbolic, elliptic, or parabolic for distance and for angle. Type II is rich enough to contain several substructures depending on where points and pencils sit relative to the real conic.


In [ ]:
classification_rows = [
    {"type": "I", "A_signature": "+,+,+", "B_signature": "+,+,+", "fundamental_object": "complex nondegenerate conic", "distance": "elliptic", "angle": "elliptic", "model_note": "elliptic plane as antipodal sphere model"},
    {"type": "II", "A_signature": "+,+,-", "B_signature": "+,+,-", "fundamental_object": "real nondegenerate conic", "distance": "mixed", "angle": "mixed", "model_note": "hyperbolic disk appears inside the conic"},
    {"type": "III", "A_signature": "+,+,0", "B_signature": "+,0,0", "fundamental_object": "two complex lines with real double point", "distance": "elliptic", "angle": "parabolic", "model_note": "dual Euclidean-type degeneration"},
    {"type": "IV", "A_signature": "+,-,0", "B_signature": "+,0,0", "fundamental_object": "two real lines with real double point", "distance": "hyperbolic", "angle": "parabolic", "model_note": "dual pseudo-Euclidean-type degeneration"},
    {"type": "V", "A_signature": "+,0,0", "B_signature": "+,+,0", "fundamental_object": "two complex absolute points on a double real line", "distance": "parabolic", "angle": "elliptic", "model_note": "Euclidean geometry with I,J at infinity"},
    {"type": "VI", "A_signature": "+,0,0", "B_signature": "+,-,0", "fundamental_object": "two real absolute points on a double real line", "distance": "parabolic", "angle": "hyperbolic", "model_note": "pseudo-Euclidean / Minkowski plane"},
    {"type": "VII", "A_signature": "+,0,0", "B_signature": "+,0,0", "fundamental_object": "double line and double point", "distance": "parabolic", "angle": "parabolic", "model_note": "Galilean-style double degeneration"},
]
classification_path = save_table(classification_rows, ARTIFACT_ROOT, "tables", "cayley-klein-classification.csv")
generated_paths.append(classification_path)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), gridspec_kw={"width_ratios": [1.4, 1.0]}, constrained_layout=True)
ax = axes[0]; ax.axis("off")
cell_text = [[row["type"], row["fundamental_object"], row["distance"], row["angle"]] for row in classification_rows]
table = ax.table(cellText=cell_text, colLabels=["type", "absolute", "distance", "angle"], cellLoc="left", colLoc="left", loc="center", colWidths=[0.08, 0.52, 0.18, 0.18])
table.auto_set_font_size(False); table.set_fontsize(7.5); table.scale(1, 1.35)
for (r, c), cell in table.get_celld().items():
    cell.set_edgecolor("#9ca3af")
    if r == 0:
        cell.set_facecolor("#e5e7eb"); cell.set_text_props(weight="bold")
ax.set_title("Seven real projective conic types", pad=14)

ax = axes[1]
labels = np.array([["IIa", "IIb", "VI"], ["IIc", "I", "V"], ["IV", "III", "VII"]], dtype=object)
ax.imshow([[0.15, 0.35, 0.55], [0.35, 0.15, 0.55], [0.75, 0.75, 0.9]], cmap="viridis", vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, labels[i, j], ha="center", va="center", fontsize=14, weight="bold", color="white" if labels[i, j] in {"IV", "III", "VII"} else "black")
ax.set_xticks(range(3), ["hyp", "ell", "par"])
ax.set_yticks(range(3), ["hyp", "ell", "par"])
ax.set_xlabel("distance measurement"); ax.set_ylabel("angle measurement")
ax.set_title("Finer measurement-type grid")
for spine in ax.spines.values():
    spine.set_visible(False)

census_path = fig_path("cayley-klein-classification-census.png")
fig.savefig(census_path, bbox_inches="tight")
plt.close(fig)
generated_paths.append(census_path)
checks.update({"classification_type_count": len(classification_rows), "classification_types": [row["type"] for row in classification_rows]})
rel(census_path), rel(classification_path)


## Model Comparison Lab

The HTML lab compares four model behaviors without using textbook figures.

- **Euclidean / type V:** the absolute is the line at infinity with complex circular points; finite distance is parabolic and must be calibrated by a unit.
- **Hyperbolic / type II interior:** the visible unit conic is a boundary at infinite real distance.
- **Elliptic / type I:** the projective plane is represented by the sphere with antipodal points identified; there is no real boundary conic.
- **Pseudo-Euclidean / type VI and degenerate limits:** real absolute directions form a light-cone structure, and angle addition has limiting directions.


In [ ]:
def make_model_comparison_html():
    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[[{"type": "xy"}, {"type": "xy"}], [{"type": "scene"}, {"type": "xy"}]],
        subplot_titles=("Euclidean: parabolic distance, elliptic angle", "Hyperbolic: type II interior of a real conic", "Elliptic: antipodal sphere model for type I", "Pseudo-Euclidean / degenerate directions"),
        horizontal_spacing=0.08,
        vertical_spacing=0.12,
    )
    grid_vals = np.linspace(-2, 2, 5)
    for v in grid_vals:
        fig.add_trace(go.Scatter(x=[-2, 2], y=[v, v], mode="lines", line=dict(color="rgba(80,80,80,0.25)", width=1), showlegend=False), row=1, col=1)
        fig.add_trace(go.Scatter(x=[v, v], y=[-2, 2], mode="lines", line=dict(color="rgba(80,80,80,0.25)", width=1), showlegend=False), row=1, col=1)
    theta = np.linspace(0, 2 * math.pi, 220)
    fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines", line=dict(color=COLORS["blue"], width=3), name="Euclidean circle"), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 0], mode="markers+text", text=["0", "unit"], textposition="bottom center", marker=dict(color="black", size=8), showlegend=False), row=1, col=1)

    fig.add_trace(go.Scatter(x=np.cos(theta), y=np.sin(theta), mode="lines", line=dict(color="black", width=3), name="absolute conic"), row=1, col=2)
    d_levels = [0, 0.6, 1.2, 1.8]
    xs = [math.tanh(d) for d in d_levels]
    fig.add_trace(go.Scatter(x=xs, y=[0] * len(xs), mode="markers+text", text=[f"d={d:g}" for d in d_levels], textposition="top center", marker=dict(color=COLORS["orange"], size=9), name="equal CK steps"), row=1, col=2)
    fig.add_trace(go.Scatter(x=[-0.95, 0.95], y=[0, 0], mode="lines", line=dict(color=COLORS["green"], width=2), showlegend=False), row=1, col=2)

    u = np.linspace(0, 2 * math.pi, 48)
    v = np.linspace(0, math.pi, 24)
    U, V = np.meshgrid(u, v)
    Xs = np.cos(U) * np.sin(V)
    Ys = np.sin(U) * np.sin(V)
    Zs = np.cos(V)
    fig.add_trace(go.Surface(x=Xs, y=Ys, z=Zs, opacity=0.32, colorscale="Blues", showscale=False, name="unit sphere"), row=2, col=1)
    t = np.linspace(0, 2 * math.pi, 250)
    fig.add_trace(go.Scatter3d(x=np.cos(t), y=np.sin(t), z=np.zeros_like(t), mode="lines", line=dict(color=COLORS["red"], width=5), name="great circle"), row=2, col=1)
    fig.add_trace(go.Scatter3d(x=np.zeros_like(t), y=np.cos(t), z=np.sin(t), mode="lines", line=dict(color=COLORS["green"], width=5), name="another line"), row=2, col=1)

    x = np.linspace(-2.2, 2.2, 400)
    fig.add_trace(go.Scatter(x=x, y=x, mode="lines", line=dict(color="black", dash="dash"), name="light direction +"), row=2, col=2)
    fig.add_trace(go.Scatter(x=x, y=-x, mode="lines", line=dict(color="black", dash="dash"), name="light direction -"), row=2, col=2)
    xp = np.linspace(1.01, 2.2, 200)
    y_h = np.sqrt(xp * xp - 1)
    fig.add_trace(go.Scatter(x=np.r_[xp, -xp[::-1]], y=np.r_[y_h, y_h[::-1]], mode="lines", line=dict(color=COLORS["purple"], width=3), name="x^2-y^2=1"), row=2, col=2)
    fig.add_trace(go.Scatter(x=np.r_[xp, -xp[::-1]], y=np.r_[-y_h, -y_h[::-1]], mode="lines", line=dict(color=COLORS["purple"], width=3), showlegend=False), row=2, col=2)

    for r, c in [(1, 1), (1, 2), (2, 2)]:
        fig.update_xaxes(range=[-2.25, 2.25], row=r, col=c)
        fig.update_yaxes(range=[-2.25, 2.25], row=r, col=c)
    fig.update_scenes(aspectmode="cube", camera=dict(eye=dict(x=1.45, y=1.45, z=0.95)))
    fig.update_layout(height=850, width=1050, title_text="Cayley-Klein model comparison from the choice of absolute", showlegend=False, margin=dict(l=40, r=40, t=80, b=40))
    path = html_path("euclidean-hyperbolic-elliptic-degenerate-comparison.html")
    fig.write_html(path, include_plotlyjs=True, full_html=True)
    return path

model_html_path = make_model_comparison_html()
generated_paths.append(model_html_path)
rel(model_html_path), model_html_path.stat().st_size


## Artifact Gallery

The notebook displays generated artifacts inline during execution. The direct links below keep the chapter readable in static form.

![I and J as absolute conic](../../artifacts/chapter-20-cayley-klein-geometries/figures/ij-revisited-absolute-conic.png)

![Line measurement trichotomy](../../artifacts/chapter-20-cayley-klein-geometries/figures/invariant-reparametrization-check.png)

![Proof dependency map](../../artifacts/chapter-20-cayley-klein-geometries/figures/visual-route-map.png)

![Planar absolute conic lab](../../artifacts/chapter-20-cayley-klein-geometries/figures/absolute-conic-measurement-lab.png)

![Cayley-Klein classification census](../../artifacts/chapter-20-cayley-klein-geometries/figures/cayley-klein-classification-census.png)

Open the interactive [model comparison lab](../../artifacts/chapter-20-cayley-klein-geometries/html/euclidean-hyperbolic-elliptic-degenerate-comparison.html), the [visual target table](../../artifacts/chapter-20-cayley-klein-geometries/tables/visual-observation-targets.csv), and the [classification table](../../artifacts/chapter-20-cayley-klein-geometries/tables/cayley-klein-classification.csv) when reading outside a live kernel.


In [ ]:
DISPLAY_ARTIFACTS = [ij_path, line_path, route_path, planar_path, census_path, model_html_path]
for artifact in DISPLAY_ARTIFACTS:
    display_artifact(artifact, width=820, height=520)


## Applied Lab: Classify a Segment Before Measuring It

Use the unit circle absolute as a local laboratory. The first task is not to compute a number; it is to classify the line through the two points. The classification tells you which branch of the measurement story you are in.

Change the point pairs in `lab_cases` and rerun the cell. A positive real cross-ratio on a secant is the ordinary hyperbolic case. A negative real cross-ratio means the segment is separated by the absolute. A negative discriminant means the absolute intersections are complex conjugates, so the line measurement is elliptic. A zero discriminant is the parabolic tangent limit.


In [ ]:
lab_cases = {
    "inside_to_inside": (np.array([-0.25, -0.15]), np.array([0.55, 0.25])),
    "inside_to_outside": (np.array([0.1, 0.15]), np.array([1.35, 0.2])),
    "outside_secant": (np.array([-1.4, -0.55]), np.array([1.3, -0.2])),
    "outside_exterior": (np.array([1.35, -0.8]), np.array([1.45, 0.95])),
    "tangent_limit": (np.array([1.0, -0.8]), np.array([1.0, 0.9])),
}
lab_rows = []
for name, (p0, q0) in lab_cases.items():
    kind, disc, cr_value = classify_line_measurement(p0, q0)
    row = {"case": name, "classification": kind, "discriminant": float(disc), "cross_ratio": None if cr_value is None else float(np.real(cr_value))}
    row["signed_distance"] = float((-0.5 * np.log(cr_value)).real) if kind == "real hyperbolic" else None
    lab_rows.append(row)
pd.DataFrame(lab_rows)


## Sanity Checks And Takeaways

The core invariant pattern is now visible.

- The absolute conic supplies the reference elements for every measurement.
- Cross-ratio multiplicativity becomes distance additivity after applying a logarithm, with the usual branch cautions.
- Hyperbolic, elliptic, and parabolic measurements are not separate formulas; they are root-type cases of the same projective construction.
- Euclidean geometry is type V: a parabolic distance scale with elliptic angle measurement from the circular points.
- Hyperbolic geometry is the interior substructure of type II, while elliptic geometry is type I and degenerate models arise when the conic pair loses rank.


In [ ]:
# Keep JSON check paths book-local; the visual audit rejects workspace-absolute paths.
png_artifacts = [ij_path, line_path, route_path, planar_path, census_path]
html_artifacts = [model_html_path]
table_artifacts = [visual_targets_path, line_samples_path, scenario_table_path, classification_path]
check_artifacts = [storyboard_path]
all_artifacts = png_artifacts + html_artifacts + table_artifacts + check_artifacts

assert_artifacts(all_artifacts, min_size=256)
assert checks["laguerre_angle_residual"] < 1e-12
assert checks["line_additivity_residual"] < 1e-12
assert checks["line_antisymmetry_residual"] < 1e-12
assert checks["elliptic_cross_ratio_unit_modulus_error"] < 1e-12
assert checks["parabolic_limit_residual"] < 1e-5
assert checks["klein_origin_formula_residual"] < 1e-12
assert checks["polar_absolute_incidence_error"] < 1e-9
assert checks["sympy_zero_distance_identity"] == "0"
assert checks["sympy_symmetry_identity"] == "0"
assert checks["sympy_additivity_identity"] == "0"
assert checks["classification_type_count"] == 7

raster_stats = []
for path in png_artifacts:
    stats = image_stats(path)
    stats["path"] = rel(path)
    raster_stats.append(stats)
    assert stats["pixel_std"] > 1.0

visual_checks = {
    "chapter": 20,
    "all_files_exist": all(path.exists() for path in all_artifacts),
    "visual_count": len(DISPLAY_ARTIFACTS),
    "cross_ratio_error": float(checks["line_additivity_residual"]),
    "raster_artifacts": raster_stats,
    "html_artifacts": [rel(path) for path in html_artifacts],
    "table_artifacts": [rel(path) for path in table_artifacts],
    "numeric_checks": {key: value for key, value in checks.items() if isinstance(value, (int, float, str, list))},
}
visual_checks_path = save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final = {
    "chapter": 20,
    "source_span": "printed pp. 375-398 / PDF pp. 397-420",
    "notebook_executed": True,
    "artifacts": [rel(path) for path in all_artifacts + [visual_checks_path]],
    "checks": visual_checks,
}
final_sanity_path = save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
assert_artifacts([visual_checks_path, final_sanity_path], min_size=256)
rel(final_sanity_path), visual_checks["numeric_checks"]
